## Shibuya Crossing Calibration Pilot

This notebook sets up a lightweight calibration workflow using publicly licensed Shibuya scramble crossing videos.

The goal is not to immediately build a perfect evacuation model, but to extract measurable pedestrian movement patterns from real footage, then use those measurements to calibrate a simple Social Force Model (SFM) baseline before moving back to emergency evacuation scenarios.

### Why Shibuya?

Shibuya Crossing is useful as a calibration pilot because it has:

- dense pedestrian movement,
- multiple crossing directions,
- many visible agents in one scene,
- publicly available videos that can be legally reused with attribution,
- a non-emergency but high-density crowd flow that is safer to use as a baseline than unverified panic assumptions.

### Important limitation

This is a movement calibration dataset, not an emergency dataset. We should use it to calibrate baseline walking behaviour such as speed, direction groups, flow, density, and spacing. We should not infer sensitive attributes such as gender from video unless there is an ethically collected labelled dataset. For simulation, it is better to use neutral agent categories such as speed class, group membership, direction, and urgency level.

### 0.1 Video Sources and Attribution

| ID | Source | Author | Year | File details | Licence | Why use it |
|---|---|---:|---:|---|---|---|
| `shibuya_2019_webm` | [Wikimedia Commons: Shibuya Crossing, Tokyo, Japan (video).webm](https://commons.wikimedia.org/wiki/File:Shibuya_Crossing,_Tokyo,_Japan_(video).webm) | Basile Morin | 2019 | 59 s, 1920 x 1080, WebM | CC BY-SA 4.0 | Higher resolution; better for tracking and scale estimation. |
| `shibuya_2009_ogv` | [Wikimedia Commons: Shibuya Scramble Crossing.ogv](https://commons.wikimedia.org/wiki/File:Shibuya_Scramble_Crossing.ogv) | Gst | 2009 | 1 min 3 s, 640 x 480, OGV | CC BY-SA 3.0 / GFDL | Older/lower-res video; useful as a second sample to test robustness. |

When reporting, cite the source page, author, licence, and mention that this notebook uses the videos only for pilot calibration of pedestrian movement metrics.

### 0.2 Calibration Plan

We will extract these measurements from each video:

| Measurement | Why it matters for SFM calibration | How we estimate it |
|---|---|---|
| Walking speed distribution | Sets desired speed `v0` and speed variance | Track pedestrian positions across frames, convert pixels to metres, compute m/s. |
| Direction groups | Helps represent crossing streams | Estimate movement angle from trajectory displacement. |
| Flow rate | Helps calibrate throughput across virtual lines | Count pedestrians crossing a measurement line per second. |
| Local density | Helps check whether simulated crowd compactness is realistic | Count agents per square metre in a calibrated region. |
| Nearest-neighbour spacing | Helps calibrate social repulsion and comfort distance | Compute pairwise distances among tracked pedestrians at each time. |

The calibration target is a baseline movement model. Emergency behaviour, panic speed, exit choice, and group behaviour should be layered after this baseline is defensible.

### 1. Setup

This cell creates local folders and defines the two source videos. By default, it does not download anything automatically. Set `DOWNLOAD_VIDEOS = True` if you want the notebook to fetch the files.

In [1]:
# numpy somehow always not installed one
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\kylet\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import math
import json
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cv2
    OPENCV_AVAILABLE = True
except ImportError:
    OPENCV_AVAILABLE = False
    print("OpenCV is not installed. Install with: pip install opencv-python")

NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data" / "shibuya"
VIDEO_DIR = DATA_DIR / "videos"
FRAME_DIR = DATA_DIR / "frames"
ANNOTATION_DIR = DATA_DIR / "annotations"
OUTPUT_DIR = DATA_DIR / "outputs"

for folder in [VIDEO_DIR, FRAME_DIR, ANNOTATION_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

VIDEOS = {
    "shibuya_2019_webm": {
        "page_url": "https://commons.wikimedia.org/wiki/File:Shibuya_Crossing,_Tokyo,_Japan_(video).webm",
        "download_url": "https://commons.wikimedia.org/wiki/Special:Redirect/file/Shibuya%20Crossing,%20Tokyo,%20Japan%20(video).webm",
        "filename": "shibuya_2019_crossing.webm",
        "author": "Basile Morin",
        "licence": "CC BY-SA 4.0",
    },
    "shibuya_2009_ogv": {
        "page_url": "https://commons.wikimedia.org/wiki/File:Shibuya_Scramble_Crossing.ogv",
        "download_url": "https://commons.wikimedia.org/wiki/Special:Redirect/file/Shibuya%20Scramble%20Crossing.ogv",
        "filename": "shibuya_2009_scramble.ogv",
        "author": "Gst",
        "licence": "CC BY-SA 3.0 / GFDL",
    },
}

DOWNLOAD_VIDEOS = False

print("Data folder:", DATA_DIR)
print("Videos to register:", list(VIDEOS.keys()))

OpenCV is not installed. Install with: pip install opencv-python
Data folder: c:\Users\kylet\HTX\Deliverable 1\Documentation\Task 1 - NTU Paper + GABM\data\shibuya
Videos to register: ['shibuya_2019_webm', 'shibuya_2009_ogv']


### 2. Download or Register Videos

If the videos are already downloaded manually, place them in `data/shibuya/videos/` using the filenames shown below. This avoids repeated downloads and keeps the notebook reproducible.

In [3]:
def download_video(video_id, overwrite=False):
    info = VIDEOS[video_id]
    out_path = VIDEO_DIR / info["filename"]

    if out_path.exists() and not overwrite:
        print(f"Already exists: {out_path}")
        return out_path

    print(f"Downloading {video_id}...")
    print(info["page_url"])
    urllib.request.urlretrieve(info["download_url"], out_path)
    print(f"Saved to {out_path}")
    return out_path

if DOWNLOAD_VIDEOS:
    for video_id in VIDEOS:
        download_video(video_id)
else:
    print("DOWNLOAD_VIDEOS is False. Manually download files or switch it to True.")
    for video_id, info in VIDEOS.items():
        print(f"{video_id}: {VIDEO_DIR / info['filename']}")

DOWNLOAD_VIDEOS is False. Manually download files or switch it to True.
shibuya_2019_webm: c:\Users\kylet\HTX\Deliverable 1\Documentation\Task 1 - NTU Paper + GABM\data\shibuya\videos\shibuya_2019_crossing.webm
shibuya_2009_ogv: c:\Users\kylet\HTX\Deliverable 1\Documentation\Task 1 - NTU Paper + GABM\data\shibuya\videos\shibuya_2009_scramble.ogv


### 3. Inspect Video Metadata

Before tracking, check that the files can be opened and record their frame rate, resolution, and duration. These values are needed when converting frame numbers into seconds.

In [4]:
def video_metadata(video_path):
    if not OPENCV_AVAILABLE:
        raise ImportError("OpenCV is required for video metadata extraction.")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "path": str(video_path),
            "exists": video_path.exists(),
            "opened": False,
            "fps": np.nan,
            "frame_count": np.nan,
            "width": np.nan,
            "height": np.nan,
            "duration_s": np.nan,
        }

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()

    duration_s = frame_count / fps if fps and fps > 0 else np.nan
    return {
        "path": str(video_path),
        "exists": video_path.exists(),
        "opened": True,
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_s": duration_s,
    }

inventory = []
for video_id, info in VIDEOS.items():
    path = VIDEO_DIR / info["filename"]
    row = {"video_id": video_id, **video_metadata(path)} if path.exists() else {
        "video_id": video_id,
        "path": str(path),
        "exists": False,
        "opened": False,
        "fps": np.nan,
        "frame_count": np.nan,
        "width": np.nan,
        "height": np.nan,
        "duration_s": np.nan,
    }
    inventory.append(row)

video_inventory = pd.DataFrame(inventory)
video_inventory

,video_id,path,exists,opened,fps,frame_count,width,height,duration_s
0,shibuya_2019_webm,c:\Users\kylet\HTX\Deliverable 1\Documentation...,False,False,NaN,NaN,NaN,NaN,NaN
1,shibuya_2009_ogv,c:\Users\kylet\HTX\Deliverable 1\Documentation...,False,False,NaN,NaN,NaN,NaN,NaN


### 4. Extract Sample Frames

For a fast pilot, extract a short window from each video, e.g. 10 seconds at 2 frames per second. This gives enough frames for manual scale calibration and a small trajectory sample without overloading the notebook.

In [5]:
def extract_frames(video_id, start_s=5, duration_s=10, sample_fps=2, overwrite=False):
    if not OPENCV_AVAILABLE:
        raise ImportError("OpenCV is required for frame extraction.")

    info = VIDEOS[video_id]
    video_path = VIDEO_DIR / info["filename"]
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    out_dir = FRAME_DIR / video_id
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    if not native_fps or native_fps <= 0:
        native_fps = 30

    start_frame = int(start_s * native_fps)
    end_frame = int((start_s + duration_s) * native_fps)
    frame_step = max(1, int(round(native_fps / sample_fps)))

    saved = []
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frame_idx = start_frame

    while frame_idx <= end_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            break

        out_path = out_dir / f"{video_id}_frame_{frame_idx:06d}.jpg"
        if overwrite or not out_path.exists():
            cv2.imwrite(str(out_path), frame)
        saved.append({"video_id": video_id, "frame": frame_idx, "time_s": frame_idx / native_fps, "path": str(out_path)})
        frame_idx += frame_step

    cap.release()
    return pd.DataFrame(saved)

# Run this after the videos exist locally.
frames_index = []
for video_id in VIDEOS:
    video_path = VIDEO_DIR / VIDEOS[video_id]["filename"]
    if video_path.exists():
        frames_index.append(extract_frames(video_id, start_s=5, duration_s=10, sample_fps=2))

frames_df = pd.concat(frames_index, ignore_index=True) if frames_index else pd.DataFrame()
frames_df.head()

""


### 5. Preview Extracted Frames

Use this to sanity-check camera angle and whether pedestrians are visible enough to annotate. If tracking is poor, choose a different 10-second window.

In [6]:
def show_frame_grid(frames_df, video_id, n=6):
    if frames_df.empty:
        print("No frames extracted yet.")
        return

    sample = frames_df[frames_df["video_id"] == video_id].head(n)
    if sample.empty:
        print(f"No frames found for {video_id}")
        return

    fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
    if len(sample) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = cv2.imread(row["path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"t={row['time_s']:.1f}s")
        ax.axis("off")

    plt.tight_layout()

for video_id in VIDEOS:
    if not frames_df.empty and video_id in frames_df["video_id"].unique():
        show_frame_grid(frames_df, video_id, n=4)

### 6. Homography / Scale Calibration

To convert pixel coordinates into real-world metres, we need a mapping from image plane to ground plane.

Pilot approach:

1. Pick one clear reference frame.
2. Select four ground points on the crossing plane.
3. Estimate their real-world coordinates in metres using known crossing markings, lane widths, or a measured map reference.
4. Compute a homography matrix.

This is the most important calibration step. Speed values are only meaningful after pixel positions are mapped into metres.

In [7]:
# Fill these after selecting four visible ground-plane points.
# image_points_px: pixel coordinates from the image, e.g. [[x1,y1], [x2,y2], ...]
# world_points_m: corresponding real-world coordinates in metres, e.g. [[0,0], [10,0], [10,8], [0,8]]

HOMOGRAPHY_CONFIG = {
    "shibuya_2019_webm": {
        "reference_frame": None,
        "image_points_px": None,
        "world_points_m": None,
        "notes": "Use the high-res video for the first calibration attempt.",
    },
    "shibuya_2009_ogv": {
        "reference_frame": None,
        "image_points_px": None,
        "world_points_m": None,
        "notes": "Lower resolution; useful as robustness check after 2019 video.",
    },
}


def compute_homography(image_points_px, world_points_m):
    if not OPENCV_AVAILABLE:
        raise ImportError("OpenCV is required for homography calibration.")

    image_points_px = np.asarray(image_points_px, dtype=np.float32)
    world_points_m = np.asarray(world_points_m, dtype=np.float32)

    if image_points_px.shape != world_points_m.shape or image_points_px.shape[0] < 4:
        raise ValueError("Need at least four matching image/world points with the same shape.")

    H, status = cv2.findHomography(image_points_px, world_points_m)
    return H, status


def pixels_to_world(points_px, H):
    points_px = np.asarray(points_px, dtype=np.float32).reshape(-1, 1, 2)
    points_m = cv2.perspectiveTransform(points_px, H)
    return points_m.reshape(-1, 2)

print("Homography functions ready. Fill HOMOGRAPHY_CONFIG before computing speeds.")

Homography functions ready. Fill HOMOGRAPHY_CONFIG before computing speeds.


### 7. Manual Annotation Template

For a defensible pilot, manually annotate a small number of pedestrians first. This is slower than automatic tracking, but it gives a clean calibration sample and helps validate whether automated tracking is believable.

Suggested pilot sample:

- 20 to 50 pedestrians per video,
- 5 to 10 seconds per video,
- annotate centre-of-body or footpoint consistently,
- record at least 5 frames per pedestrian,
- avoid pedestrians who are heavily occluded.

CSV schema:

| Column | Meaning |
|---|---|
| `video_id` | Which source video. |
| `ped_id` | Unique pedestrian ID within the video. |
| `frame` | Frame number in the video. |
| `time_s` | Timestamp in seconds. |
| `x_px` | Pixel x-coordinate. |
| `y_px` | Pixel y-coordinate. |
| `visibility` | `visible`, `partial`, or `occluded`. |

In [8]:
ANNOTATION_COLUMNS = ["video_id", "ped_id", "frame", "time_s", "x_px", "y_px", "visibility"]

for video_id in VIDEOS:
    template_path = ANNOTATION_DIR / f"{video_id}_manual_tracks.csv"
    if not template_path.exists():
        pd.DataFrame(columns=ANNOTATION_COLUMNS).to_csv(template_path, index=False)
        print("Created:", template_path)
    else:
        print("Already exists:", template_path)

Created: c:\Users\kylet\HTX\Deliverable 1\Documentation\Task 1 - NTU Paper + GABM\data\shibuya\annotations\shibuya_2019_webm_manual_tracks.csv
Created: c:\Users\kylet\HTX\Deliverable 1\Documentation\Task 1 - NTU Paper + GABM\data\shibuya\annotations\shibuya_2009_ogv_manual_tracks.csv


### 8. Load Annotations and Convert to Real-World Coordinates

Once the manual CSV files have points, this cell converts pixel tracks into metre coordinates using the homography matrix.

If homography is not filled yet, the cell still loads the raw annotations so we can check whether the CSV format is correct.

In [9]:
def load_manual_annotations():
    rows = []
    for video_id in VIDEOS:
        path = ANNOTATION_DIR / f"{video_id}_manual_tracks.csv"
        if path.exists():
            df = pd.read_csv(path)
            if not df.empty:
                rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=ANNOTATION_COLUMNS)


def add_world_coordinates(tracks_px, homography_config):
    if tracks_px.empty:
        return tracks_px.copy()

    output = []
    for video_id, group in tracks_px.groupby("video_id"):
        cfg = homography_config.get(video_id, {})
        img_pts = cfg.get("image_points_px")
        world_pts = cfg.get("world_points_m")

        g = group.copy()
        if img_pts is None or world_pts is None:
            g["x_m"] = np.nan
            g["y_m"] = np.nan
            g["homography_ready"] = False
        else:
            H, _ = compute_homography(img_pts, world_pts)
            xy_m = pixels_to_world(g[["x_px", "y_px"]].to_numpy(), H)
            g["x_m"] = xy_m[:, 0]
            g["y_m"] = xy_m[:, 1]
            g["homography_ready"] = True
        output.append(g)

    return pd.concat(output, ignore_index=True)

tracks_px = load_manual_annotations()
tracks_world = add_world_coordinates(tracks_px, HOMOGRAPHY_CONFIG)
tracks_world.head()

,video_id,ped_id,frame,time_s,x_px,y_px,visibility


### 9. Compute Walking Speeds and Direction Groups

This cell computes speed from real-world trajectories.

Formula:

`speed = distance travelled / time difference`

Direction is estimated from movement angle. This lets us identify streams such as left-to-right, right-to-left, diagonal, etc.

In [10]:
def compute_track_kinematics(tracks_world, min_dt=1e-6):
    if tracks_world.empty:
        return tracks_world.copy()

    required = {"video_id", "ped_id", "time_s", "x_m", "y_m"}
    missing = required - set(tracks_world.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = tracks_world.sort_values(["video_id", "ped_id", "time_s"]).copy()
    df["prev_x_m"] = df.groupby(["video_id", "ped_id"])["x_m"].shift(1)
    df["prev_y_m"] = df.groupby(["video_id", "ped_id"])["y_m"].shift(1)
    df["prev_time_s"] = df.groupby(["video_id", "ped_id"])["time_s"].shift(1)

    df["dx_m"] = df["x_m"] - df["prev_x_m"]
    df["dy_m"] = df["y_m"] - df["prev_y_m"]
    df["dt_s"] = df["time_s"] - df["prev_time_s"]
    df.loc[df["dt_s"] <= min_dt, "dt_s"] = np.nan

    df["step_distance_m"] = np.sqrt(df["dx_m"] ** 2 + df["dy_m"] ** 2)
    df["speed_mps"] = df["step_distance_m"] / df["dt_s"]
    df["direction_deg"] = np.degrees(np.arctan2(df["dy_m"], df["dx_m"]))

    return df

kinematics = compute_track_kinematics(tracks_world)
kinematics[["video_id", "ped_id", "time_s", "x_m", "y_m", "speed_mps", "direction_deg"]].head()

KeyError: "['x_m', 'y_m', 'speed_mps', 'direction_deg'] not in index"

### 10. Summarise Observed Movement Metrics

These are the headline values to compare against the SFM baseline.

For reporting, focus on:

- mean walking speed,
- standard deviation of walking speed,
- 10th, 50th, and 90th percentile speed,
- number of tracked pedestrians,
- number of usable speed observations.

In [ ]:
def summarise_speeds(kinematics):
    if kinematics.empty or "speed_mps" not in kinematics.columns:
        return pd.DataFrame()

    clean = kinematics.replace([np.inf, -np.inf], np.nan).dropna(subset=["speed_mps"])
    clean = clean[(clean["speed_mps"] >= 0) & (clean["speed_mps"] <= 4.0)]

    if clean.empty:
        return pd.DataFrame()

    summary = clean.groupby("video_id").agg(
        n_pedestrians=("ped_id", "nunique"),
        n_speed_obs=("speed_mps", "count"),
        mean_speed_mps=("speed_mps", "mean"),
        std_speed_mps=("speed_mps", "std"),
        p10_speed_mps=("speed_mps", lambda x: np.percentile(x, 10)),
        median_speed_mps=("speed_mps", "median"),
        p90_speed_mps=("speed_mps", lambda x: np.percentile(x, 90)),
    ).reset_index()

    return summary

speed_summary = summarise_speeds(kinematics)
speed_summary

In [ ]:
if not kinematics.empty and "speed_mps" in kinematics.columns:
    clean = kinematics.replace([np.inf, -np.inf], np.nan).dropna(subset=["speed_mps"])
    clean = clean[(clean["speed_mps"] >= 0) & (clean["speed_mps"] <= 4.0)]

    if not clean.empty:
        plt.figure(figsize=(8, 4))
        for video_id, group in clean.groupby("video_id"):
            plt.hist(group["speed_mps"], bins=20, alpha=0.5, label=video_id)
        plt.xlabel("walking speed (m/s)")
        plt.ylabel("count")
        plt.title("Observed walking speed distribution")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()
    else:
        print("No valid speed observations yet.")
else:
    print("No annotations loaded yet.")

### 11. Flow Across a Measurement Line

Flow is useful because SFM should reproduce not only individual speed but also collective throughput.

Define a virtual line in world coordinates, then count how many unique pedestrians cross it per time bin.

In [ ]:
def compute_line_flow(kinematics, video_id, axis="x", line_value=0.0, bin_s=1.0):
    if kinematics.empty:
        return pd.DataFrame(columns=["video_id", "time_bin_s", "crossings", "flow_p_per_s"])

    df = kinematics[kinematics["video_id"] == video_id].copy()
    coord = f"{axis}_m"
    prev_coord = f"prev_{axis}_m"

    if coord not in df.columns or prev_coord not in df.columns:
        raise ValueError(f"Missing {coord} or {prev_coord}")

    before = df[prev_coord] < line_value
    after = df[coord] >= line_value
    crossed_forward = before & after

    before_reverse = df[prev_coord] > line_value
    after_reverse = df[coord] <= line_value
    crossed_reverse = before_reverse & after_reverse

    crossings = df[crossed_forward | crossed_reverse].copy()
    if crossings.empty:
        return pd.DataFrame(columns=["video_id", "time_bin_s", "crossings", "flow_p_per_s"])

    crossings["time_bin_s"] = (crossings["time_s"] // bin_s) * bin_s
    out = crossings.groupby("time_bin_s").agg(crossings=("ped_id", "nunique")).reset_index()
    out["video_id"] = video_id
    out["flow_p_per_s"] = out["crossings"] / bin_s
    return out[["video_id", "time_bin_s", "crossings", "flow_p_per_s"]]

# Example after homography is ready:
# flow_df = compute_line_flow(kinematics, "shibuya_2019_webm", axis="x", line_value=5.0, bin_s=1.0)
# flow_df

### 12. Density and Nearest-Neighbour Spacing

Density and spacing help calibrate social-force interaction strength. If simulated agents are too packed or too spread out compared with the video, the social force parameters are probably wrong.

In [ ]:
def nearest_neighbour_spacing(tracks_world, time_round_s=0.5):
    if tracks_world.empty:
        return pd.DataFrame()

    df = tracks_world.dropna(subset=["x_m", "y_m"]).copy()
    if df.empty:
        return pd.DataFrame()

    df["time_bin_s"] = (df["time_s"] / time_round_s).round() * time_round_s
    rows = []

    for (video_id, t), group in df.groupby(["video_id", "time_bin_s"]):
        points = group[["x_m", "y_m"]].to_numpy()
        ped_ids = group["ped_id"].to_numpy()
        if len(points) < 2:
            continue

        diff = points[:, None, :] - points[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.nan)
        nn = np.nanmin(dist, axis=1)

        for ped_id, spacing in zip(ped_ids, nn):
            rows.append({"video_id": video_id, "time_bin_s": t, "ped_id": ped_id, "nn_spacing_m": spacing})

    return pd.DataFrame(rows)

spacing_df = nearest_neighbour_spacing(tracks_world)
spacing_df.head()

### 13. SFM Calibration Targets

After extracting measurements, we can calibrate the SFM baseline against these targets.

| Parameter | Suggested pilot range | Meaning |
|---|---:|---|
| `desired_speed_mean` | 0.9 to 1.8 m/s | Average preferred walking speed. |
| `desired_speed_std` | 0.1 to 0.5 m/s | Variation between pedestrians. |
| `max_speed_multiplier` | 1.1 to 1.8 | Upper bound relative to desired speed. |
| `relaxation_time` | 0.3 to 1.0 s | How quickly agents adapt velocity. |
| `social_force.factor` | 2.0 to 10.0 | Strength of agent-agent repulsion. |
| `social_force.gamma` | 0.2 to 0.8 | Angular/interaction falloff parameter. |
| `obstacle_force.factor` | 5.0 to 20.0 | Strength of wall/obstacle repulsion. |
| `agent_radius` | 0.25 to 0.40 m | Personal body radius / occupied space. |

For Shibuya, the first calibration target should be normal dense walking, not panic. After that, emergency speed multipliers can be tested separately.

In [ ]:
PARAM_BOUNDS = pd.DataFrame([
    {"parameter": "desired_speed_mean", "low": 0.9, "high": 1.8, "unit": "m/s"},
    {"parameter": "desired_speed_std", "low": 0.1, "high": 0.5, "unit": "m/s"},
    {"parameter": "max_speed_multiplier", "low": 1.1, "high": 1.8, "unit": "ratio"},
    {"parameter": "relaxation_time", "low": 0.3, "high": 1.0, "unit": "s"},
    {"parameter": "social_force.factor", "low": 2.0, "high": 10.0, "unit": "model"},
    {"parameter": "social_force.gamma", "low": 0.2, "high": 0.8, "unit": "model"},
    {"parameter": "obstacle_force.factor", "low": 5.0, "high": 20.0, "unit": "model"},
    {"parameter": "agent_radius", "low": 0.25, "high": 0.40, "unit": "m"},
])

PARAM_BOUNDS

### 14. Calibration Objective Function

This cell defines how we compare observed video measurements against simulated measurements.

A good first objective is a weighted error over:

- mean speed,
- speed standard deviation,
- median speed,
- mean flow,
- nearest-neighbour spacing.

This keeps calibration simple and explainable.

In [ ]:
def metric_summary_from_tracks(kinematics, spacing_df=None, flow_df=None):
    metrics = {}

    if kinematics is not None and not kinematics.empty and "speed_mps" in kinematics.columns:
        speed = kinematics["speed_mps"].replace([np.inf, -np.inf], np.nan).dropna()
        speed = speed[(speed >= 0) & (speed <= 4.0)]
        if not speed.empty:
            metrics["mean_speed_mps"] = float(speed.mean())
            metrics["std_speed_mps"] = float(speed.std())
            metrics["median_speed_mps"] = float(speed.median())

    if spacing_df is not None and not spacing_df.empty and "nn_spacing_m" in spacing_df.columns:
        spacing = spacing_df["nn_spacing_m"].replace([np.inf, -np.inf], np.nan).dropna()
        if not spacing.empty:
            metrics["mean_nn_spacing_m"] = float(spacing.mean())

    if flow_df is not None and not flow_df.empty and "flow_p_per_s" in flow_df.columns:
        metrics["mean_flow_p_per_s"] = float(flow_df["flow_p_per_s"].mean())

    return metrics


def calibration_loss(observed_metrics, simulated_metrics, weights=None):
    if weights is None:
        weights = {
            "mean_speed_mps": 2.0,
            "std_speed_mps": 1.0,
            "median_speed_mps": 1.0,
            "mean_nn_spacing_m": 1.0,
            "mean_flow_p_per_s": 2.0,
        }

    total = 0.0
    used = []
    for key, weight in weights.items():
        if key in observed_metrics and key in simulated_metrics:
            obs = observed_metrics[key]
            sim = simulated_metrics[key]
            scale = max(abs(obs), 1e-6)
            err = ((sim - obs) / scale) ** 2
            total += weight * err
            used.append(key)

    return {"loss": total, "metrics_used": used}

observed_metrics = metric_summary_from_tracks(kinematics, spacing_df=spacing_df, flow_df=None)
observed_metrics

### 15. Parameter Search Skeleton

This is the scaffold for calibration. The missing piece is `simulate_sfm_with_params(params)`, which should run the chosen SFM implementation and return the same metrics as the observed video.

For speed, start with random search or a small grid rather than a heavy optimiser.

In [ ]:
def sample_parameter_sets(param_bounds, n=20, seed=42):
    rng = np.random.default_rng(seed)
    samples = []

    for i in range(n):
        row = {"sample_id": i}
        for _, p in param_bounds.iterrows():
            row[p["parameter"]] = rng.uniform(p["low"], p["high"])
        samples.append(row)

    return pd.DataFrame(samples)


def simulate_sfm_with_params(params):
    """
    TODO: connect this to the project's SFM runner.

    Expected return format:
    {
        "mean_speed_mps": ...,
        "std_speed_mps": ...,
        "median_speed_mps": ...,
        "mean_nn_spacing_m": ...,
        "mean_flow_p_per_s": ...,
    }

    Keep this separate from the observed-data pipeline so we can swap in
    baseline / behavioural / group models later.
    """
    raise NotImplementedError("Connect this function to the SFM simulation runner.")

candidate_params = sample_parameter_sets(PARAM_BOUNDS, n=10, seed=1)
candidate_params.head()

### 16. How This Connects Back to FYP.ipynb

The original `FYP.ipynb` models evacuation behaviour directly. This Shibuya notebook adds a validation step before emergency experiments:

| Notebook | Purpose | Main output |
|---|---|---|
| `FYP.ipynb` | Existing evacuation simulation with baseline, behavioural, and group models. | Evacuation performance under scenario assumptions. |
| `Research.ipynb` | Literature comparison and paper-inspired proxy experiments. | Justification of methods and limitations. |
| `Shibuya_Calibration.ipynb` | Real video calibration for normal dense pedestrian movement. | Calibrated baseline movement parameters before adding emergency behaviour. |

This means the project story becomes stronger:

1. Literature shows SFM and related models are commonly used.
2. Paper-inspired tests show how scenarios affect evacuation outcomes.
3. Shibuya video calibration gives a real-world movement baseline.
4. Emergency/stadium simulations can then be described as scenario extensions from a calibrated baseline, not arbitrary parameter choices.

### 17. Immediate Next Steps

1. Download both Wikimedia videos into `data/shibuya/videos/` or set `DOWNLOAD_VIDEOS = True`.
2. Extract a 10-second frame sample from each video.
3. Pick one frame from `shibuya_2019_webm` and define four homography points.
4. Manually annotate 20 to 50 pedestrians across 5 to 10 seconds.
5. Run speed, direction, flow, and spacing summaries.
6. Wire `simulate_sfm_with_params(params)` to the existing SFM code.
7. Compare observed metrics against simulated metrics and choose a calibrated baseline parameter set.

For the supervisor discussion, the key point is: this notebook separates real-world calibration from emergency scenario experimentation, which makes the later stadium model more defensible.